In [ ]:
import sys, os, distutils.core
# Note: This is a faster way to install detectron2 in Colab, but it does not include all functionalities (e.g. compiled operators).
# See https://detectron2.readthedocs.io/tutorials/install.html for full installation instructions
!git clone 'https://github.com/facebookresearch/detectron2'
dist = distutils.core.run_setup("./detectron2/setup.py")
!python -m pip install {' '.join([f"'{x}'" for x in dist.install_requires])}
sys.path.insert(0, os.path.abspath('./detectron2'))

In [ ]:
!pip install ultralytics

In [ ]:
import torch
import cv2
import numpy as np
import json
import os
import time
import yaml
import pandas as pd
from tqdm import tqdm
from pycocotools.coco import COCO
from pycocotools import mask as mask_utils

# --- LIBRARY MODEL ---
from ultralytics import YOLO
from detectron2.engine import DefaultPredictor
from detectron2.config import get_cfg
from detectron2 import model_zoo

# ================= KONFIGURASI =================
DETECTRON2_TYPE = 'Detectron2_R50_FPN'
YOLO_TYPE = 'YOLOv12_Medium'
ANNOTATION_MODE = "simplified" # single_class, '',simplified
IMG_SIZE = 960
IOU_THRESHOLD = 0.5  # Batas minimal dibilang "Benar"
CONF_THRESHOLD = 0.5 # Batas kepercayaan model


# Path Dataset & Model (SESUAIKAN INI!)
VERSION = "C_2026_1d80_10_10_AUG"
DATASET_ROOT = f'/content/drive/MyDrive/Colab Notebooks/TA_CuttingRockDescription/datasets/{VERSION}'
annotation_postfix = f"_{ANNOTATION_MODE}" if ANNOTATION_MODE != '' else ''
TEST_JSON = f"{DATASET_ROOT}/test/test_annotations{annotation_postfix}.json" # Gunakan JSON single class
TEST_IMG_DIR = f"{DATASET_ROOT}/test/images"

# Path Model Final Anda
YOLO_MODEL_PATH = f"{DATASET_ROOT}/models/{YOLO_TYPE}/weights/best.pt"
DETECTRON_MODEL_DIR = f"{DATASET_ROOT}/models/{DETECTRON2_TYPE}"
DETECTRON_WEIGHTS = f"{DETECTRON_MODEL_DIR}/model_final.pth"
# ===============================================

data_yaml_path = f"{DATASET_ROOT}/data.yaml"
with open(data_yaml_path, 'r') as f:
    data_yaml = yaml.safe_load(f)
NUM_CLASSES = data_yaml.get('nc', len(data_yaml['names']))
if ANNOTATION_MODE == "single_class":
    NUM_CLASSES = 1
print(f"Classes: {NUM_CLASSES}")

def calculate_iou_mask(mask1, mask2):
    intersection = np.logical_and(mask1, mask2).sum()
    union = np.logical_or(mask1, mask2).sum()
    if union == 0: return 0
    return intersection / union

class IndependentEvaluator:
    def __init__(self):
        print("⚡ Menyiapkan Independent Evaluator + Speed Test...")
        self.coco = COCO(TEST_JSON)
        self.img_ids = self.coco.getImgIds()

        # Load YOLO
        print("🤖 Loading YOLO...")
        self.yolo = YOLO(YOLO_MODEL_PATH)

        # Load Detectron2
        print("🤖 Loading Detectron2...")
        cfg = get_cfg()
        cfg.merge_from_file(model_zoo.get_config_file("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"))
        cfg.MODEL.ROI_HEADS.NUM_CLASSES = NUM_CLASSES
        cfg.INPUT.MIN_SIZE_TEST = IMG_SIZE
        cfg.INPUT.MAX_SIZE_TEST = 1333
        cfg.MODEL.WEIGHTS = DETECTRON_WEIGHTS
        cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = CONF_THRESHOLD
        cfg.TEST.DETECTIONS_PER_IMAGE = 1000
        self.detectron = DefaultPredictor(cfg)

    def get_gt_masks(self, img_id, height, width):
        ann_ids = self.coco.getAnnIds(imgIds=img_id)
        anns = self.coco.loadAnns(ann_ids)
        masks = []
        for ann in anns:
            mask = self.coco.annToMask(ann)
            masks.append(mask)
        return np.array(masks) if masks else np.zeros((0, height, width))

    def evaluate(self, limit=None):
        print(f"\n🚀 Memulai Evaluasi Independen (Akurasi & Kecepatan)...")

        metrics = {
            'YOLO': {'TP': 0, 'FP': 0, 'FN': 0, 'Total_IoU': 0, 'Count': 0, 'Total_Time': 0},
            'Detectron2': {'TP': 0, 'FP': 0, 'FN': 0, 'Total_IoU': 0, 'Count': 0, 'Total_Time': 0}
        }

        img_list = self.img_ids[:limit] if limit else self.img_ids

        # WARMUP GPU (Penting agar perhitungan waktu tidak bias di awal)
        print("🔥 Warming up GPU...")
        dummy_img = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        self.yolo.predict(dummy_img, verbose=False)
        self.detectron(dummy_img)

        for img_id in tqdm(img_list):
            img_info = self.coco.loadImgs(img_id)[0]
            img_path = os.path.join(TEST_IMG_DIR, img_info['file_name'])
            image = cv2.imread(img_path)
            h, w = image.shape[:2]
            gt_masks = self.get_gt_masks(img_id, h, w)

            # --- 1. RUN YOLO (WITH TIMER) ---
            # Sync CUDA sebelum start waktu
            if torch.cuda.is_available(): torch.cuda.synchronize()
            t_start = time.time()

            yolo_res = self.yolo.predict(image, imgsz=IMG_SIZE, conf=CONF_THRESHOLD, verbose=False, max_det=1000)[0]

            # Sync CUDA setelah end waktu
            if torch.cuda.is_available(): torch.cuda.synchronize()
            t_end = time.time()
            metrics['YOLO']['Total_Time'] += (t_end - t_start) # Detik

            # Process YOLO Result
            if yolo_res.masks is not None:
                yolo_masks = yolo_res.masks.data.cpu().numpy()
                yolo_masks_resized = [cv2.resize(m, (w, h), interpolation=cv2.INTER_NEAREST) for m in yolo_masks]
                yolo_masks = np.array(yolo_masks_resized)
            else:
                yolo_masks = np.zeros((0, h, w))

            self._match_and_score(gt_masks, yolo_masks, metrics['YOLO'])

            # --- 2. RUN DETECTRON2 (WITH TIMER) ---
            if torch.cuda.is_available(): torch.cuda.synchronize()
            t_start = time.time()

            d2_res = self.detectron(image)

            if torch.cuda.is_available(): torch.cuda.synchronize()
            t_end = time.time()
            metrics['Detectron2']['Total_Time'] += (t_end - t_start) # Detik

            # Process D2 Result
            d2_instances = d2_res["instances"]
            if len(d2_instances) > 0:
                d2_masks = d2_instances.pred_masks.cpu().numpy()
            else:
                d2_masks = np.zeros((0, h, w))

            self._match_and_score(gt_masks, d2_masks, metrics['Detectron2'])

        return self._calculate_final_metrics(metrics, len(img_list))

    def _match_and_score(self, gt_masks, pred_masks, metric_dict):
        # (Logika matching sama seperti sebelumnya)
        matched_gt = set()
        for p_mask in pred_masks:
            best_iou = 0
            best_gt_idx = -1
            for g_idx, g_mask in enumerate(gt_masks):
                if g_idx in matched_gt: continue
                iou = calculate_iou_mask(p_mask, g_mask)
                if iou > best_iou:
                    best_iou = iou
                    best_gt_idx = g_idx

            if best_iou >= IOU_THRESHOLD:
                metric_dict['TP'] += 1
                metric_dict['Total_IoU'] += best_iou
                metric_dict['Count'] += 1
                matched_gt.add(best_gt_idx)
            else:
                metric_dict['FP'] += 1
        metric_dict['FN'] += (len(gt_masks) - len(matched_gt))

    def _calculate_final_metrics(self, metrics, num_images):
        summary = []
        for model_name, data in metrics.items():
            tp, fp, fn = data['TP'], data['FP'], data['FN']

            precision = tp / (tp + fp) if (tp + fp) > 0 else 0
            recall = tp / (tp + fn) if (tp + fn) > 0 else 0
            f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
            mean_iou = data['Total_IoU'] / data['Count'] if data['Count'] > 0 else 0

            # Speed Metrics
            total_time_ms = data['Total_Time'] * 1000
            avg_latency = total_time_ms / num_images
            fps = 1000 / avg_latency if avg_latency > 0 else 0

            summary.append({
                "Model": model_name,
                "FPS": round(fps, 2),                  # <--- Metric Baru
                "Latency (ms)": round(avg_latency, 2), # <--- Metric Baru
                "F1-Score": round(f1 * 100, 2),        # Pengganti Simple untuk mAP
                "Precision": round(precision * 100, 2),
                "Recall": round(recall * 100, 2),
                "Avg Pixel IoU": round(mean_iou * 100, 2),
                "TP": tp, "FP": fp, "FN": fn
            })
        return pd.DataFrame(summary)

# --- JALANKAN ---
evaluator = IndependentEvaluator()
df_results = evaluator.evaluate(limit=None) # Set limit=10 untuk tes cepat
df_results.to_csv(f"{DATASET_ROOT}/models/independent_evaluation_results_{DETECTRON2_TYPE}_vs_{YOLO_TYPE}.csv", index=False)

print("\n🏆 HASIL FINAL: INDEPENDENT BENCHMARK (SPEED + ACCURACY) 🏆")
print(df_results.to_markdown(index=False)) # Format Markdown biar rapi

In [ ]:
import torch
import cv2
import numpy as np
import json
import os
import time
import yaml
import pandas as pd
import matplotlib.pyplot as plt

# --- LIBRARY MODEL ---
from ultralytics import YOLO, SAM
from detectron2.engine import DefaultPredictor
from detectron2.config import get_cfg
from detectron2 import model_zoo

# ================= KONFIGURASI (Ensure these paths are correct for your setup) =================
DETECTRON2_TYPE = 'Detectron2_R50_FPN'
YOLO_TYPE = 'YOLOv12_Medium_Tuned'
ANNOTATION_MODE = "simplified" # single_class, '',simplified
IMG_SIZE = 960
IOU_THRESHOLD = 0.3  # Batas minimal dibilang "Benar"
CONF_THRESHOLD = 0.3 # Batas kepercayaan model


# Path Dataset & Model (SESUAIKAN INI!)
VERSION = "C_2026_1d80_10_10_AUG"
DATASET_ROOT = f'/content/drive/MyDrive/Colab Notebooks/TA_CuttingRockDescription/datasets/{VERSION}'
annotation_postfix = f"_{ANNOTATION_MODE}" if ANNOTATION_MODE != '' else ''
TEST_JSON = f"{DATASET_ROOT}/test/test_annotations{annotation_postfix}.json" # Gunakan JSON single class
TEST_IMG_DIR = f"{DATASET_ROOT}/test/images"

# Path Model Final Anda
YOLO_MODEL_PATH = f"{DATASET_ROOT}/models/{YOLO_TYPE}/weights/best.pt"
DETECTRON_MODEL_DIR = f"{DATASET_ROOT}/models/{DETECTRON2_TYPE}"
DETECTRON_WEIGHTS = f"{DETECTRON_MODEL_DIR}/model_final.pth"
# ===============================================

data_yaml_path = f"{DATASET_ROOT}/data.yaml"
with open(data_yaml_path, 'r') as f:
    data_yaml = yaml.safe_load(f)
NUM_CLASSES = data_yaml.get('nc', len(data_yaml['names']))
if ANNOTATION_MODE == "single_class":
    NUM_CLASSES = 1
print(f"Classes: {NUM_CLASSES}")

# --- 1. Mount Google Drive ---
# print("Mounting Google Drive...")
# drive.mount('/content/drive')

# --- 2. Initialize Models ---
print("\nInitializing YOLO, Detectron2, and MobileSAM models...")

# Initialize YOLO
yolo_model = YOLO(YOLO_MODEL_PATH)

# Initialize MobileSAM
mobile_sam_model = SAM("mobile_sam.pt")

# Initialize Detectron2
cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"))
cfg.MODEL.ROI_HEADS.NUM_CLASSES = NUM_CLASSES
cfg.INPUT.MIN_SIZE_TEST = IMG_SIZE
cfg.INPUT.MAX_SIZE_TEST = 1333
cfg.MODEL.WEIGHTS = DETECTRON_WEIGHTS
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = CONF_THRESHOLD
cfg.TEST.DETECTIONS_PER_IMAGE = 1000
detectron_predictor = DefaultPredictor(cfg)

print("Models initialized and ready for inference.")

# --- 3. Get the image path from the user ---
image_path = input("\nPlease enter the full path to the image on Google Drive (e.g., /content/drive/MyDrive/path/to/your/image.jpg): ")

# --- 4. Validate and load the image ---
if not os.path.exists(image_path):
    print(f"\nError: Image not found at '{image_path}'. Please check the path and ensure Google Drive is mounted correctly.")
else:
    print(f"Loading image from: '{image_path}'")
    image = cv2.imread(image_path)
    if image is None:
        print(f"\nError: Could not load image from '{image_path}'. It might be corrupted or not a valid image file.")
    else:
        h, w = image.shape[:2]
        print(f"Image loaded with dimensions: {w}x{h}")

        # --- Run YOLO Prediction ---
        print("Running YOLO prediction...")
        yolo_results = yolo_model.predict(image, imgsz=IMG_SIZE, conf=CONF_THRESHOLD, verbose=False, max_det=1000)[0]
        yolo_masks_display = []
        if yolo_results.masks is not None:
            yolo_masks_data = yolo_results.masks.data.cpu().numpy()
            for m in yolo_masks_data:
                resized_mask = cv2.resize(m.astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST)
                yolo_masks_display.append(resized_mask > 0.5)
        yolo_masks_display = np.array(yolo_masks_display)
        print(f"YOLO detected {len(yolo_masks_display)} masks.")

        # --- Run Detectron2 Prediction ---
        print("Running Detectron2 prediction...")
        d2_results = detectron_predictor(image)
        d2_masks_display = []
        d2_instances = d2_results["instances"]
        if len(d2_instances) > 0:
            d2_masks_data = d2_instances.pred_masks.cpu().numpy()
            for m in d2_masks_data:
                d2_masks_display.append(m > 0.5)
        d2_masks_display = np.array(d2_masks_display)
        print(f"Detectron2 detected {len(d2_masks_display)} masks.")

        # --- Run MobileSAM Prediction ---
        print("Running MobileSAM prediction...")
        sam_results = mobile_sam_model(image, verbose=False)[0]
        sam_masks_display = []
        if sam_results.masks is not None:
            sam_masks_data = sam_results.masks.data.cpu().numpy()
            for m in sam_masks_data:
                resized_mask = cv2.resize(m.astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST)
                sam_masks_display.append(resized_mask > 0.5)
        sam_masks_display = np.array(sam_masks_display)
        print(f"MobileSAM detected {len(sam_masks_display)} masks.")

        # --- Visualization ---
        # Helper function to overlay masks on an image with random colors
        def overlay_masks(original_img, masks):
            display_img = cv2.cvtColor(original_img.copy(), cv2.COLOR_BGR2RGB)
            if len(masks) > 0:
                for i, mask in enumerate(masks):
                    color = np.random.rand(3)  # Random color for each mask
                    colored_overlay = np.zeros_like(display_img, dtype=np.uint8)
                    colored_overlay[mask] = (np.array(color) * 255).astype(np.uint8)
                    display_img = cv2.addWeighted(display_img, 1, colored_overlay, 0.5, 0)
            return display_img

        fig, axes = plt.subplots(2, 2, figsize=(20, 20)) # 2x2 grid for 4 images
        axes = axes.flatten()

        # Original Image
        axes[0].imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        axes[0].set_title("Original Image", fontsize=16)
        axes[0].axis('off')

        # YOLO Segmentation
        yolo_viz_img = overlay_masks(image, yolo_masks_display)
        axes[1].imshow(yolo_viz_img)
        axes[1].set_title(f"YOLO Segmentation ({len(yolo_masks_display)} masks)", fontsize=16)
        axes[1].axis('off')

        # Detectron2 Segmentation
        d2_viz_img = overlay_masks(image, d2_masks_display)
        axes[2].imshow(d2_viz_img)
        axes[2].set_title(f"Detectron2 Segmentation ({len(d2_masks_display)} masks)", fontsize=16)
        axes[2].axis('off')

        # MobileSAM Segmentation
        sam_viz_img = overlay_masks(image, sam_masks_display)
        axes[3].imshow(sam_viz_img)
        axes[3].set_title(f"MobileSAM Segmentation ({len(sam_masks_display)} masks)", fontsize=16)
        axes[3].axis('off')

        plt.tight_layout()
        plt.show()


In [ ]:
import torch
import cv2
import numpy as np
import json
import os
import time
import yaml
import pandas as pd
import matplotlib.pyplot as plt

# --- LIBRARY MODEL ---
from ultralytics import YOLO, SAM
from detectron2.engine import DefaultPredictor
from detectron2.config import get_cfg
from detectron2 import model_zoo

# ================= KONFIGURASI (Ensure these paths are correct for your setup) =================
DETECTRON2_TYPE = 'Detectron2_R50_FPN'
YOLO_TYPE = 'YOLOv12_Medium_Tuned'
ANNOTATION_MODE = "simplified" # single_class, '',simplified
IMG_SIZE = 960
IOU_THRESHOLD = 0.3  # Batas minimal dibilang "Benar"
CONF_THRESHOLD = 0.3 # Batas kepercayaan model


# Path Dataset & Model (SESUAIKAN INI!)
VERSION = "C_2026_1d80_10_10_AUG"
DATASET_ROOT = f'/content/drive/MyDrive/Colab Notebooks/TA_CuttingRockDescription/datasets/{VERSION}'
annotation_postfix = f"_{ANNOTATION_MODE}" if ANNOTATION_MODE != '' else ''
TEST_JSON = f"{DATASET_ROOT}/test/test_annotations{annotation_postfix}.json" # Gunakan JSON single class
TEST_IMG_DIR = f"{DATASET_ROOT}/test/images"

# Path Model Final Anda
YOLO_MODEL_PATH = f"{DATASET_ROOT}/models/{YOLO_TYPE}/weights/best.pt"
DETECTRON_MODEL_DIR = f"{DATASET_ROOT}/models/{DETECTRON2_TYPE}"
DETECTRON_WEIGHTS = f"{DETECTRON_MODEL_DIR}/model_final.pth"
# ===============================================

data_yaml_path = f"{DATASET_ROOT}/data.yaml"
with open(data_yaml_path, 'r') as f:
    data_yaml = yaml.safe_load(f)
NUM_CLASSES = data_yaml.get('nc', len(data_yaml['names']))
if ANNOTATION_MODE == "single_class":
    NUM_CLASSES = 1
print(f"Classes: {NUM_CLASSES}")

# --- 1. Mount Google Drive ---
# print("Mounting Google Drive...")
# drive.mount('/content/drive')

# --- 2. Initialize Models ---
print("\nInitializing YOLO, Detectron2, and MobileSAM models...")

# Initialize YOLO
yolo_model = YOLO(YOLO_MODEL_PATH)

# Initialize MobileSAM
mobile_sam_model = SAM("mobile_sam.pt")

# Initialize Detectron2
cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"))
cfg.MODEL.ROI_HEADS.NUM_CLASSES = NUM_CLASSES
cfg.INPUT.MIN_SIZE_TEST = IMG_SIZE
cfg.INPUT.MAX_SIZE_TEST = 1333
cfg.MODEL.WEIGHTS = DETECTRON_WEIGHTS
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = CONF_THRESHOLD
cfg.TEST.DETECTIONS_PER_IMAGE = 1000
detectron_predictor = DefaultPredictor(cfg)

print("Models initialized and ready for inference.")

# --- 3. Get the image path from the user ---
image_path = input("\nPlease enter the full path to the image on Google Drive (e.g., /content/drive/MyDrive/path/to/your/image.jpg): ")

# --- 4. Validate and load the image ---
if not os.path.exists(image_path):
    print(f"\nError: Image not found at '{image_path}'. Please check the path and ensure Google Drive is mounted correctly.")
else:
    print(f"Loading image from: '{image_path}'")
    image = cv2.imread(image_path)
    if image is None:
        print(f"\nError: Could not load image from '{image_path}'. It might be corrupted or not a valid image file.")
    else:
        h, w = image.shape[:2]
        print(f"Image loaded with dimensions: {w}x{h}")

        # --- Run YOLO Prediction ---
        print("Running YOLO prediction...")
        yolo_results = yolo_model.predict(image, imgsz=IMG_SIZE, conf=CONF_THRESHOLD, verbose=False, max_det=1000)[0]
        yolo_masks_display = []
        if yolo_results.masks is not None:
            yolo_masks_data = yolo_results.masks.data.cpu().numpy()
            for m in yolo_masks_data:
                resized_mask = cv2.resize(m.astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST)
                yolo_masks_display.append(resized_mask > 0.5)
        yolo_masks_display = np.array(yolo_masks_display)
        print(f"YOLO detected {len(yolo_masks_display)} masks.")

        # --- Run Detectron2 Prediction ---
        print("Running Detectron2 prediction...")
        d2_results = detectron_predictor(image)
        d2_masks_display = []
        d2_instances = d2_results["instances"]
        if len(d2_instances) > 0:
            d2_masks_data = d2_instances.pred_masks.cpu().numpy()
            for m in d2_masks_data:
                d2_masks_display.append(m > 0.5)
        d2_masks_display = np.array(d2_masks_display)
        print(f"Detectron2 detected {len(d2_masks_display)} masks.")

        # --- Run YOLO + MobileSAM Hybrid (Custom Model) ---
        # Use YOLO bounding boxes as prompts for MobileSAM refinement
        print("Running YOLO + MobileSAM Hybrid prediction...")
        hybrid_masks_display = []
        if yolo_results.boxes is not None and len(yolo_results.boxes) > 0:
            # Extract YOLO bounding boxes as prompts for SAM
            yolo_bboxes = yolo_results.boxes.xyxy.cpu().numpy()  # [x1, y1, x2, y2] format
            # Run SAM with YOLO bboxes as prompts (single SAM call)
            hybrid_results = mobile_sam_model(image, bboxes=yolo_bboxes, verbose=False)[0]
            if hybrid_results.masks is not None:
                hybrid_masks_data = hybrid_results.masks.data.cpu().numpy()
                for m in hybrid_masks_data:
                    resized_mask = cv2.resize(m.astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST)
                    hybrid_masks_display.append(resized_mask > 0.5)
        hybrid_masks_display = np.array(hybrid_masks_display)
        print(f"YOLO+SAM Hybrid detected {len(hybrid_masks_display)} masks.")

        # --- Visualization ---
        # Helper function to overlay masks on an image with random colors
        def overlay_masks(original_img, masks):
            display_img = cv2.cvtColor(original_img.copy(), cv2.COLOR_BGR2RGB)
            if len(masks) > 0:
                for i, mask in enumerate(masks):
                    color = np.random.rand(3)  # Random color for each mask
                    colored_overlay = np.zeros_like(display_img, dtype=np.uint8)
                    colored_overlay[mask] = (np.array(color) * 255).astype(np.uint8)
                    display_img = cv2.addWeighted(display_img, 1, colored_overlay, 0.5, 0)
            return display_img

        fig, axes = plt.subplots(2, 2, figsize=(20, 20)) # 2x2 grid for 4 images
        axes = axes.flatten()

        # Original Image
        axes[0].imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        axes[0].set_title("Original Image", fontsize=16)
        axes[0].axis('off')

        # YOLO Segmentation
        yolo_viz_img = overlay_masks(image, yolo_masks_display)
        axes[1].imshow(yolo_viz_img)
        axes[1].set_title(f"YOLO Segmentation ({len(yolo_masks_display)} masks)", fontsize=16)
        axes[1].axis('off')

        # Detectron2 Segmentation
        d2_viz_img = overlay_masks(image, d2_masks_display)
        axes[2].imshow(d2_viz_img)
        axes[2].set_title(f"Detectron2 Segmentation ({len(d2_masks_display)} masks)", fontsize=16)
        axes[2].axis('off')

        # YOLO + MobileSAM Hybrid (Custom Model)
        hybrid_viz_img = overlay_masks(image, hybrid_masks_display)
        axes[3].imshow(hybrid_viz_img)
        axes[3].set_title(f"YOLO+SAM Hybrid ({len(hybrid_masks_display)} masks)", fontsize=16)
        axes[3].axis('off')

        plt.tight_layout()
        plt.show()
